# Cleanup Export Volume

Clears **all contents** of the export volume so the next export run starts clean. Does not touch registered models.

## Configuration

Read widget parameters: source catalog, schema, and volume name. Cleans the entire volume root.

In [ ]:
dbutils.widgets.text("source_catalog", "", "Source catalog (holds model_exports)")
dbutils.widgets.text("source_schema", "default", "Source schema")
dbutils.widgets.text("volume_name", "model_exports", "Volume name")

SOURCE_CATALOG = dbutils.widgets.get("source_catalog").strip()
SOURCE_SCHEMA = dbutils.widgets.get("source_schema").strip()
VOLUME_NAME = dbutils.widgets.get("volume_name").strip()

if not SOURCE_CATALOG:
    raise ValueError("Set source_catalog")

VOLUME_PATH = f"/Volumes/{SOURCE_CATALOG}/{SOURCE_SCHEMA}/{VOLUME_NAME}"
print(f"Cleaning entire volume: {VOLUME_PATH}")

## Remove export files

Recursively deletes all files and directories in the export volume. Skips gracefully if the volume is empty or path does not exist.

In [ ]:
def path_not_found(e):
    msg = str(e).lower()
    return (
        "path does not exist" in msg
        or "cannot find" in msg
        or "cannot be found" in msg
        or "no such file" in msg
        or "filenotfound" in msg
        or "not_found" in msg
        or "does not exist" in msg
    )

try:
    entries = dbutils.fs.ls(VOLUME_PATH)
    for entry in entries:
        dbutils.fs.rm(entry.path, recurse=True)
        print(f"  Removed: {entry.path}")
    print(f"Cleaned: {VOLUME_PATH} ({len(entries)} items)")
except Exception as e:
    if path_not_found(e):
        print(f"Skip (not found): {VOLUME_PATH}")
    else:
        raise
print("Export cleanup done.")